<a href="https://colab.research.google.com/github/SawantSameer/csot-ml-astronomy/blob/main/Submissions/SameerSawant/week2/task2_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSoT'26 - ML in Astronomy - Week 2 . Part 2: Your First Neural Network (Starter)

**Goal:** Define a 2-layer fully-connected network (MLP) with `nn.Module`, forward-pass a real batch, set up a loss + optimiser, and run a single optimisation step to watch the loss drop.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).
2. Read [`04-building-models-with-nn-module.md`](../04-building-models-with-nn-module.md) and [`05-loss-functions-and-optimisers.md`](../05-loss-functions-and-optimisers.md).

Replace each `TODO` with working code. **Do not** open the solution until you've genuinely attempted every TODO. (We *set up* training here; the full multi-epoch training loop is Week 3.)

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [5]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [7]:
# TODO: paste your Week 1 data pipeline here so that the following names are defined:
#   train_ds, val_ds, test_ds, train_loader, val_loader, test_loader, num_classes
#
# The quickest path is to copy the data-prep cells from
# ../../Week-1/notebooks/week1_data_solution.ipynb (Steps 1-8), then add:
#   num_classes = len(train_ds.classes)

# TODO: paste your Week 1 data pipeline here so that the following names are defined:
#   train_ds, val_ds, test_ds, train_loader, val_loader, test_loader, num_classes
#
# The quickest path is to copy the data-prep cells from
# ../../Week-1/notebooks/week1_data_solution.ipynb (Steps 1-8), then add:
#   num_classes = len(train_ds.classes)

import os
import json
import shutil
from pathlib import Path
from torchvision import transforms # Added this line
from torchvision.datasets import ImageFolder # Added this line

# --- 1. KAGGLE AUTHENTICATION ---
# Replace with your EXACT username and key from your Kaggle account
kaggle_creds = {
  "username": "your_username_here",
  "key": "your_long_alphanumeric_key_here"
}

# Create the hidden Kaggle directory and write the credentials file
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

# Lock down the file permissions for security
!chmod 600 /root/.kaggle/kaggle.json

# --- 2. DIRECTORY SETUP ---
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"
DATA_ROOT = Path("galaxy_data")
RAW_ROOT.mkdir(parents=True, exist_ok=True)

# --- 3. DOWNLOAD KAGGLE IMAGES ---
print("Authenticating and downloading images from Kaggle. This might take a minute...")
dataset_name = "jaimetrickz/galaxy-zoo-2-images"
!kaggle datasets download -d {dataset_name} -p {RAW_ROOT} --unzip

# --- 4. DOWNLOAD HART LABELS ---
print("Downloading Hart et al. morphology labels...")
!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz
!gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

# --- 5. FIX NESTED DIRECTORY ---
# Kaggle sometimes double-wraps the extracted folder. This flattens it.
nested_dir = IMAGES_DIR / "images_gz2"

if nested_dir.exists() and nested_dir.is_dir():
    print("Flattening the nested image directory...")
    for file in nested_dir.glob("*.jpg"):
        shutil.move(str(file), str(IMAGES_DIR))
    nested_dir.rmdir() # Clean up the empty duplicate folder

import shutil
from pathlib import Path

RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"

print("Hunting for the hidden images...")

# Recursively find ALL jpg files anywhere inside RAW_ROOT
all_jpgs = list(RAW_ROOT.rglob("*.jpg"))
print(f"Found {len(all_jpgs)} total images hiding in the raw folder!")

if len(all_jpgs) > 0:
    print("Moving them to the correct IMAGES_DIR... (This might take a minute)")

    # Ensure IMAGES_DIR exists
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)

    # Move them all to the flat directory
    moved_count = 0
    for img_path in all_jpgs:
        # Only move if it's not already in the right spot
        if img_path.parent != IMAGES_DIR:
            shutil.move(str(img_path), str(IMAGES_DIR / img_path.name))
            moved_count += 1

    print(f"Successfully moved {moved_count} images!")

# Final Recount
print("\n--- Final Image Recount ---")
image_count = len(list(IMAGES_DIR.glob('*.jpg')))
print(f"Total images exactly in target folder: {image_count}")

if image_count > 240000:
    print("✅ Found them! Data pipeline setup successful! Ready for Step 2.")
else:
    print("⚠️ Warning: Something is still odd. Let me know what it prints!")



# --- 6. FINAL SANITY CHECK ---
print("\n--- Final Download Check ---")
print(f"Mapping CSV exists: {(RAW_ROOT / 'gz2_filename_mapping.csv').exists()}")
print(f"Hart labels CSV exists: {(RAW_ROOT / 'gz2_hart16.csv').exists()}")
print(f"Images folder exists: {IMAGES_DIR.exists()}")

if IMAGES_DIR.exists():
    image_count = len(list(IMAGES_DIR.glob('*.jpg')))
    print(f"Total images found: {image_count}")
    if image_count > 240000:
        print("✅ Data pipeline setup successful! Ready for Step 2.")
    else:
        print("⚠️ Warning: Image count seems low.")

# TODO: print os.listdir(RAW_ROOT) and count a few files in IMAGES_DIR.
#       Hint: Path(IMAGES_DIR).glob("*.jpg")
import pandas as pd
from pathlib import Path

# Assuming these are still set from the previous cell
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"

print("--- 1. Contents of RAW_ROOT ---")
# List everything sitting in the raw folder
for item in RAW_ROOT.iterdir():
    print(f" - {item.name}")

print("\n--- 2. Peek at IMAGES_DIR ---")
# Grab just the first 5 jpg files to verify the naming convention
sample_images = list(IMAGES_DIR.glob('*.jpg'))[:5]
for img in sample_images:
    print(f" - {img.name}")

print("\n--- 3. Checking gz2_filename_mapping.csv ---")
mapping_csv_path = RAW_ROOT / "gz2_filename_mapping.csv"

if mapping_csv_path.exists():
    # Load the CSV into a pandas DataFrame
    df_mapping = pd.read_csv(mapping_csv_path)

    print("Columns found:")
    print(df_mapping.columns.tolist())

    print("\nFirst 3 rows:")
    # Display the first few rows to see the actual data
    display(df_mapping.head(3))

    # Programmatic check for the specific columns
    expected_cols = {'objid', 'sample', 'asset_id'}
    actual_cols = set(df_mapping.columns)

    if expected_cols.issubset(actual_cols):
        print("\n✅ Success: Expected columns (objid, sample, asset_id) are all present!")
    else:
        print("\n⚠️ Warning: Missing expected columns.")
else:
    print(f"⚠️ Error: Could not find {mapping_csv_path}")


print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

print("\nMapping CSV preview:")
print(pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3))

print("\nLabels CSV preview (note dr7objid — we rename to objid before merging):")
print(pd.read_csv(RAW_ROOT / "gz2_hart16.csv", nrows=3)[["dr7objid", "gz2_class"]])



Authenticating and downloading images from Kaggle. This might take a minute...
Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [02:29<00:00, 22.0MB/s]

Hunting for the hidden images...
Found 243434 total images hiding in the raw folder!
Moving them to the correct IMAGES_DIR... (This might take a minute)
Successfully moved 243434 images!

--- Final Image Recount ---
Total images exactly in target folder: 243434
✅ Found them! Data pipeline setup successful! Ready for Step 2.

--- Final Download Check ---
Mapping CSV exists: True
Hart labels CSV exists: True
Images folder exists: True
Total images found: 243434
✅ Data pipeline setup successful! Ready for Step 2.
--- 1. Contents of RAW_ROOT ---
 - gz2_hart16.csv
 - images_gz2
 - gz2_filename_mapping.csv

--- 2. Peek at IMAGES_DIR ---
 - 254459.jpg
 - 279278.jpg
 - 266304.jpg
 - 118531.jpg
 - 54582.jpg

--- 3. Checking gz2_filename_mapping.cs

,objid,sample,asset_id
0,587722981736120347,original,1
1,587722981736579107,original,2
2,587722981741363294,original,3



✅ Success: Expected columns (objid, sample, asset_id) are all present!
RAW_ROOT contents: ['gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2: 243,434

Mapping CSV preview:
                objid    sample  asset_id
0  587722981736120347  original         1
1  587722981736579107  original         2
2  587722981741363294  original         3

Labels CSV preview (note dr7objid — we rename to objid before merging):
             dr7objid gz2_class
0  587732591714893851      Sc+t
1  588009368545984617      Sb+t
2  587732484359913515        Ei


In [13]:
def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, …) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None  # artifact / ambiguous
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    """Join Kaggle mapping (objid ↔ asset_id) with Hart et al. morphology labels."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    return df


def _link_image(src: Path, dst: Path) -> bool:
    """Symlink if possible; otherwise copy (some Drive setups block symlinks)."""
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src.resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(
    images_dir,
    df,
    out_root,
    per_class=200,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """Create out_root/{train,val,test}/<class>/*.jpg for ImageFolder."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    images_dir = Path(images_dir)
    out_root = Path(out_root)
    summary = {}

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)

        n = len(rows)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        n_test = n - n_train - n_val
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train : n_train + n_val],
            "test": rows.iloc[n_train + n_val :],
        }

        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

df = load_labeled_table(
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
)
print("Joined rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nExample rows:")
print(df[["asset_id", "objid", "gz2_class", "label"]].head())


PER_CLASS = 200  # increase once the pipeline works (e.g. 2000)

summary = build_split_imagefolder_layout(
    IMAGES_DIR,
    df,
    DATA_ROOT,
    per_class=PER_CLASS,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
)
print("Linked images per class and split:")
print(pd.DataFrame(summary).fillna(0).astype(int))

for split in ("train", "val", "test"):
    split_dir = DATA_ROOT / split
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir()) if split_dir.exists() else []
    n_imgs = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"{split:5s}: {n_imgs:4d} images  classes={classes}")


transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}  n={len(ds):4d}  classes={ds.classes}")

print("class_to_idx:", train_ds.class_to_idx)


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print("train batch images:", images.shape)   # (32, 3, 64, 64)
print("train batch labels:", labels.shape)     # (32,)
num_classes = len(train_ds.classes)



Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0         3  587722981741363294        Sb      spiral
1         4  587722981741363323      Sc?l      spiral
2         5  587722981741559888        Er  elliptical
3         6  587722981741625481      Sc1t      spiral
4         7  587722981741625484        Sb      spiral
Linked images per class and split:
       elliptical  spiral  spiral_barred
train           0       0              0
val             0       0              0
test            0       0              0
train:  420 images  classes=['elliptical', 'spiral', 'spiral_barred']
val  :   90 images  classes=['elliptical', 'spiral', 'spiral_barred']
test :   90 images  classes=['elliptical', 'spiral', 'spiral_barred']
train  n= 420  classes=['elliptical', 'spiral', 'spiral_barred']
val    n=  90  classes=['elliptical', 'spiral', 'sp

## Step 1 - Define the model

A 2-layer MLP: flatten -> Linear(12288, 128) -> ReLU -> Linear(128, num_classes). The output layer returns **raw logits** (no softmax - `CrossEntropyLoss` adds it). Don't forget `super().__init__()`.

In [9]:
# TODO: define GalaxyMLP(nn.Module) with:
#   self.flatten = nn.Flatten()
#   self.fc1 = nn.Linear(3*64*64, 128); self.relu = nn.ReLU()
#   self.fc2 = nn.Linear(128, num_classes)
# and a forward(self, x) that runs flatten -> fc1 -> relu -> fc2 and returns logits.
import torch
import torch.nn as nn

class GalaxyMLP(nn.Module):
    def __init__(self, in_features=3*64*64, hidden=128, num_classes=3):
        super().__init__()                       # MUST come first
        self.flatten = nn.Flatten()              # (B,3,64,64) -> (B,12288)
        self.fc1 = nn.Linear(in_features, hidden)  # first dense layer
        self.relu = nn.ReLU()                    # non-linearity
        self.fc2 = nn.Linear(hidden, num_classes)  # output layer

    def forward(self, x):                        # x: (B, 3, 64, 64)
        x = self.flatten(x)                      # (B, 12288)
        x = self.fc1(x)                          # (B, 128)
        x = self.relu(x)                         # (B, 128)
        x = self.fc2(x)                          # (B, num_classes)  <- logits
        return x

## Step 2 - Instantiate and move to the device

Use the real `num_classes` from your data, and `.to(device)` so the model lives where the batches will.

In [10]:
# TODO: model = GalaxyMLP(num_classes=num_classes).to(device)
model = GalaxyMLP(num_classes=3).to(device)

## Step 3 - Inspect the architecture

`print(model)` shows the layers; counting `.parameters()` shows how many weights there are. Notice that the first `Linear` (12288 x 128) dominates - a direct cost of flattening.

In [11]:
# TODO: print(model), then print total and trainable parameter counts.

print(model)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
Total parameters     : 1,573,379
Trainable parameters : 1,573,379


## Step 4 - Forward-pass one real batch

Pull a batch from `train_loader`, move it to the device, and run it through the model. The output should be logits of shape `(batch_size, num_classes)`.

In [23]:
# TODO: images, labels = next(iter(train_loader)); move both to device.
#       logits = model(images); print logits.shape and confirm it's (B, num_classes).
mages, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device).long()
logits = model(images).to(device)
print(logits.shape)

torch.Size([32, 3])


## Step 5 - Loss and optimiser

`CrossEntropyLoss` consumes raw logits + integer labels. `Adam` with `lr=1e-3` is the sensible default.

In [28]:
# TODO: criterion = nn.CrossEntropyLoss()
#       optimizer = optim.Adam(model.parameters(), lr=1e-3)

import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()         # raw logits, shape (B, num_classes)
loss = criterion(logits, labels) # labels: shape (B,), dtype torch.long

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)


## Step 6 - Sanity-check the starting loss

For an untrained model on `C` balanced classes the loss should sit near `ln(C)`. If it's wildly off (or `nan`), suspect label dtype, a stray softmax, or unnormalised inputs.

In [29]:
# TODO: loss = criterion(logits, labels); print loss.item() and compare to math.log(num_classes).
loss = criterion(logits, labels)
print (loss.item())

1.1033388376235962


## Step 7 - One optimisation step (learning, in miniature)

The three lines at the heart of every training loop: clear gradients, backprop, update. Re-evaluate the loss on the *same* batch afterwards - it should drop. (Week 3 repeats this over all batches for many epochs.)

In [35]:
model.train()                          # training mode
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

logits = model(images)                 # forward
loss = criterion(logits, labels)       # how wrong
print("loss before step:", loss.item())

optimizer.zero_grad()                  # clear
loss.backward()                        # gradients
optimizer.step()                       # update

logits2 = model(images)                # forward again on same batch
print("loss after step :", criterion(logits2, labels).item())

loss before step: 3.84169864654541
loss after step : 2.6459903717041016


## Step 8 (stretch) - How big does the model get?

Re-build the MLP with different hidden widths and print the parameter count each time. The first `Linear` (in_features x hidden) dominates - flattening is expensive. A CNN (Week 3) does more with far fewer parameters by sharing weights across the image.

In [ ]:
# TODO (optional): rebuild the MLP with hidden widths 64/128/256/512 and print param counts.

## Reflection *(write 2-3 sentences each)*

1. Your untrained loss should have been near `ln(num_classes)`. Why is that the expected starting value?
2. After one step the loss dropped on that batch. Why is that *not yet* the same as 'the model is trained'?
3. Compare the MLP's parameter count to what you'd expect a CNN to need. Why does flattening make the first layer so large?

*(Replace this prompt with your answers.)*